In [1]:
import pandas as pd
print(pd.__version__)

3.0.2


In [5]:
import os
os.getcwd()

'c:\\Users\\anton\\projets\\projet1_aenord\\notebooks'

In [6]:
territoire = pd.read_csv('../data/liste_territoire.csv')
territoire.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/liste_territoire.csv'

In [7]:
import os
os.listdir('../data')

['financement.csv', 'liste_territoire.csv.csv', 'operation.csv']

In [6]:
import pandas as pd
territoire = pd.read_csv('../data/liste_territoire.csv')
territoire['insee_ardt'] = territoire['insee_ardt'].astype(str).str[:-2]
territoire.head()


,sirene,libelle_territoire,type_territoire,insee_ardt,insee_siege,sirene_epci,num_circonscription,lib_ardt
0,200027159,Dunkerque,COM,594,59183.0,245900428.0,13.0,Dunkerque
1,200030633,CA du Caudrésis et du Catésis,EPCI,592,59063.0,200030633.0,12.0,Cambrai
2,200036895,SIECF,SI,594,59295.0,NaN,15.0,Dunkerque
3,200040947,CC de Flandre Intérieure,EPCI,594,59295.0,200040947.0,15.0,Dunkerque
4,200040954,CC des Hauts de Flandre,EPCI,594,59067.0,200040954.0,14.0,Dunkerque


In [7]:
operation = pd.read_csv('../data/operation.csv')
operation.head()

,id_operation,sirene,intitule_projet,programmation,statut,cout_ht,date_integration
0,1,200036895,"Renouvellement de l’éclairage public LED, réno...",2020,NaN,"43725,14",02/02/2023 12:00:00
1,2,215906561,aménagement hôtel de police municipale,2023,NaN,92962,30/11/2023 00:00:00
2,3,200057123,construction groupe scolaire et salle multisport,2023,NaN,0,30/11/2023 00:00:00
3,4,215900689,Rénovation énergétique des locaux de l’ancienn...,2023,NaN,106061,30/11/2023 00:00:00
4,5,215906389,Changement des portes et fenêtres de la garder...,2023,NaN,40761,30/11/2023 00:00:00


In [ ]:
financement = pd.read_csv('../data/financement.csv', sep=";")
financements_actifs=financement[financement['is_annule'] != True]
financements_actifs.head()
financement['type_fct'].unique()

def attribution_typefinancement(type_fct):
    if 'friche'in type_fct.lower():
     return "RF"
    elif 'vert'in type_fct.lower():
     return 'FV'
    elif 'dsil' in type_fct.lower():
     return 'DSIL'
    elif 'detr'in type_fct.lower():
     return 'DETR'
    else:
      return "Ya un problème mec!"

financements_actifs['fund_type'] = financements_actifs['type_fct'].apply(attribution_typefinancement)
financements_actifs.head()
financements_actifs['fund_type'].unique()

<StringArray>
['FV', 'DSIL', 'RF', 'DETR']
Length: 4, dtype: str

In [63]:
financements_operations = financements_actifs.merge(operation[['id_operation', 'sirene','intitule_projet','programmation']], on='id_operation', how='left')
print(financements_operations)

          id_fct              type_fct          EJ  id_operation  montant_sub  \
0     1406966052        Fonds vert 380  1406966052          1965      75800.0   
1     2012791864  DSIL 119 ? classique  2012791864          1391     100000.0   
2     2101866885  DSIL 119 ? classique  2101866885          1389    1469403.0   
3     2101866886  DSIL 119 ? classique  2101866886          1388     290042.0   
4     2101866887  DSIL 119 ? classique  2101866887          1387      99448.0   
...          ...                   ...         ...           ...          ...   
2848  2104677241                  DETR  2104677241          2894      31498.0   
2849  2104677243                  DETR  2104677243          2895      70842.0   
2850  2104692278                  DETR  2104692278          2896     200000.0   
2851  2104677309                  DETR  2104677309          2897     200000.0   
2852  2104701698                  DETR  2104701698          2898      84845.0   

     date_ac_recp  date_not

In [67]:
financements_operations=financements_operations.merge(territoire[['sirene','insee_ardt']], on='sirene', how='left')
financements_operations['insee_ardt'].isna().sum()

np.int64(7)

In [74]:
sub_ardt = financements_operations.groupby(['insee_ardt','programmation', 'fund_type'])['montant_sub'].sum()
sub_ardt=sub_ardt.reset_index()
print(sub_ardt)

    insee_ardt  programmation fund_type  montant_sub
0          591           2016      DSIL    2499403.0
1          591           2017      DSIL    1200159.0
2          591           2018      DSIL    2142279.0
3          591           2019      DSIL    1940284.0
4          591           2020      DSIL    4610865.0
..         ...            ...       ...          ...
120        596           2025        FV    2215955.0
121        596           2025        RF    1000000.0
122        999           2023        FV    7231418.0
123        999           2024        FV    1218090.0
124        999           2025        FV      75800.0

[125 rows x 4 columns]


In [83]:
print(sub_ardt[(sub_ardt['insee_ardt'] == '594') & (sub_ardt['programmation'].isin([2021, 2022, 2023, 2024, 2025]))]['montant_sub'].sum())

39392612.0


In [87]:
subs = financements_operations.groupby(['sirene', 'programmation', 'fund_type'])['montant_sub'].sum()
subs = subs.reset_index()
print(subs)

         sirene  programmation fund_type  montant_sub
0     200027159           2016      DSIL     217333.0
1     200027159           2017      DSIL     215000.0
2     200027159           2018      DSIL     107443.0
3     200027159           2019      DSIL     123342.0
4     200027159           2020      DSIL     219720.0
...         ...            ...       ...          ...
2353  892944786           2023      DETR      91176.0
2354  908369283           2024        RF    1026302.0
2355  922232533           2025        RF     150000.0
2356  979906112           2024        RF     162286.0
2357  982438939           2024        RF     500000.0

[2358 rows x 4 columns]
